# Superstore Dataset - Exploratory Data Analysis (EDA)
**Project**: Ask Your Data — AI Analytics Copilot  
**Phase**: Phase 2 — Exploratory Data Analysis  
**Environment**: Python 3.11 (`AI-Anaylytics-env`)

---

## 1. Objective of this Notebook
Before loading data into a SQL database or building an AI copilot, we must thoroughly understand the raw data:
1. **Schema & Column Names**: What business attributes exist?
2. **Data Types**: Are dates, numbers, and categories stored in suitable formats?
3. **Data Quality**: Are there missing values (nulls) or duplicate records?
4. **Distributions & Outliers**: Are profits negative? What is the date range?
5. **Database Normalization Needs**: How should column names and data types be standardized for SQLite?


In [1]:
import os
import pandas as pd
import numpy as np

# Configure Pandas display options for readable outputs
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")


Pandas Version: 3.0.6
NumPy Version:  2.4.6


## 2. Loading the Raw Dataset
We load `Sample - Superstore.csv` from the project root. Notice that we do not alter or overwrite the original file.


In [2]:
dataset_path = os.path.join("..", "Sample - Superstore.csv")

# Read CSV with Windows encoding fallback for special characters
df = pd.read_csv(dataset_path, encoding='windows-1252')
print(f"Dataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(5)


Dataset Dimensions: 9,994 rows x 21 columns


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 3. Column Data Types & Memory Usage
Pandas assigns a data type (`dtype`) to each column.
* `int64` / `float64`: Numeric values.
* `object`: Text strings or mixed values.
Notice that `Order Date` and `Ship Date` are initially detected as `object` (strings), not timestamps.


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

## 4. Missing Values & Duplicate Records Check
In data engineering, null values and duplicate rows can cause incorrect SQL aggregations or faulty joins.


In [4]:
missing_counts = df.isna().sum()
duplicate_rows = df.duplicated().sum()

print("--- Missing Values per Column ---")
if missing_counts.sum() == 0:
    print("No missing values found in any column!")
else:
    print(missing_counts[missing_counts > 0])

print(f"\nTotal exact duplicate rows: {duplicate_rows}")


--- Missing Values per Column ---
No missing values found in any column!

Total exact duplicate rows: 0


## 5. Statistical Summary of Numeric Metrics
Let's analyze the business metrics: `Sales`, `Quantity`, `Discount`, and `Profit`.
* Is `Profit` always positive? (Look at `min`)
* What is the range of `Discount`?


In [5]:
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe().T


,count,mean,std,min,25%,50%,75%,max
Sales,9994.0,229.858001,623.245101,0.444,17.28000,54.4900,209.940,22638.480
Quantity,9994.0,3.789574,2.225110,1.000,2.00000,3.0000,5.000,14.000
Discount,9994.0,0.156203,0.206452,0.000,0.00000,0.2000,0.200,0.800
Profit,9994.0,28.656896,234.260108,-6599.978,1.72875,8.6665,29.364,8399.976


## 6. Categorical Attributes & Cardinality
Let's check the distinct count (cardinality) of key business dimensions:
* `Category` and `Sub-Category`
* `Segment`
* `Region` and `Ship Mode`


In [6]:
categorical_cols = ['Category', 'Sub-Category', 'Segment', 'Region', 'Ship Mode']
for col in categorical_cols:
    print(f"=== {col} (Unique count: {df[col].nunique()}) ===")
    print(df[col].value_counts().to_dict())
    print()


=== Category (Unique count: 3) ===
{'Office Supplies': 6026, 'Furniture': 2121, 'Technology': 1847}

=== Sub-Category (Unique count: 17) ===
{'Binders': 1523, 'Paper': 1370, 'Furnishings': 957, 'Phones': 889, 'Storage': 846, 'Art': 796, 'Accessories': 775, 'Chairs': 617, 'Appliances': 466, 'Labels': 364, 'Tables': 319, 'Envelopes': 254, 'Bookcases': 228, 'Fasteners': 217, 'Supplies': 190, 'Machines': 115, 'Copiers': 68}

=== Segment (Unique count: 3) ===
{'Consumer': 5191, 'Corporate': 3020, 'Home Office': 1783}

=== Region (Unique count: 4) ===
{'West': 3203, 'East': 2848, 'Central': 2323, 'South': 1620}

=== Ship Mode (Unique count: 4) ===
{'Standard Class': 5968, 'Second Class': 1945, 'First Class': 1538, 'Same Day': 543}



## 7. Date Format & Timeline Analysis
In SQL databases (like SQLite), dates are best stored in ISO-8601 standard format (`YYYY-MM-DD`) so they can be filtered, indexed, and sorted lexicographically.
Let's inspect the date range in the Superstore dataset.


In [7]:
order_dates = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
print(f"Earliest Order Date: {order_dates.min().strftime('%Y-%m-%d')}")
print(f"Latest Order Date:   {order_dates.max().strftime('%Y-%m-%d')}")
print(f"Total Date Span:     {(order_dates.max() - order_dates.min()).days} days (~4 years)")


Earliest Order Date: 2014-01-03
Latest Order Date:   2017-12-30
Total Date Span:     1457 days (~4 years)


## 8. Summary of Findings & Next Steps (Phase 3 Preparation)

### Key Findings:
1. **Row Count**: Exactly 9,994 transaction records across 21 columns.
2. **Data Cleanliness**: Zero null values across all columns and zero exact duplicate rows.
3. **Negative Profits**: Profits range from -$6,599.98 to +$8,399.98. The AI copilot will need to understand that profit can be negative (losses).
4. **Dates**: Raw format is `M/D/YYYY`. SQLite doesn't have a native `DATE` storage class; it stores dates as ISO text (`YYYY-MM-DD`). We must standardize them during ETL in Phase 3.
5. **Column Names**: Column names have spaces and hyphens (e.g., `Order Date`, `Sub-Category`, `Row ID`). In SQL, spaces require square brackets `[Order Date]` or quotes. In Phase 3, we should convert column names into standard `snake_case` (`order_date`, `sub_category`, `sales`, etc.) for seamless, clean SQL generation.
